# Advanced Problems with Solutions: Updating, Merging, and Copying Python Dictionaries

This notebook develops the ideas in the supplied lesson—`dict.update`, dictionary unpacking, collision behavior, insertion order, shallow copies, and deep copies—into progressively harder engineering problems.

## Learning goals

By the end, you should be able to:

- update dictionaries from mappings, pair iterables, and keyword arguments;
- reason precisely about collision rules and key order;
- build non-mutating and transactional merge utilities;
- choose between shallow copy, deep copy, and copy-on-write;
- safely forward configuration dictionaries as keyword arguments;
- audit layered configuration changes;
- handle cycles and shared references with `deepcopy`;
- benchmark equivalent copying approaches responsibly;
- combine these ideas in a production-style configuration resolver.

## Recommended workflow

1. Read each problem and its constraints.
2. Hide or skip the solution cells and implement your own version.
3. Run the provided tests.
4. Compare your implementation with the reference solution and explanation.

All solutions use the Python standard library only.


## 0. Setup and compact refresher

The source lesson shows three input forms for `dict.update`:

1. another mapping;
2. an iterable of `(key, value)` pairs;
3. keyword arguments.

It also establishes two important rules:

- on a collision, the later value replaces the earlier value;
- replacing a value does not move an existing key to a new insertion position.

The modern dictionary union operators are included as an extension:

- `left | right` creates a new shallowly merged dictionary;
- `left |= right` updates `left` in place.


In [1]:
from __future__ import annotations

from collections.abc import Callable, Iterable, Mapping
from copy import deepcopy
from dataclasses import dataclass
from inspect import Parameter, signature
from statistics import median
from timeit import repeat
from types import MappingProxyType
from typing import Any


def heading(title: str) -> None:
    print(f"\n{'=' * len(title)}\n{title}\n{'=' * len(title)}")


In [2]:
# Three forms of dict.update
example = {"a": 1, "b": 2}
example.update({"b": 20, "c": 30})
example.update([("d", 40), ("e", 50)])
example.update(f=60)

assert example == {"a": 1, "b": 20, "c": 30, "d": 40, "e": 50, "f": 60}
assert list(example) == ["a", "b", "c", "d", "e", "f"]
example


{'a': 1, 'b': 20, 'c': 30, 'd': 40, 'e': 50, 'f': 60}

In [3]:
# Collision behavior and order
left = {"first": 1, "shared": "left", "last": 3}
right = {"shared": "right", "new": 4}

merged_unpacking = {**left, **right}
merged_union = left | right

assert merged_unpacking == merged_union
assert merged_union["shared"] == "right"
assert list(merged_union) == ["first", "shared", "last", "new"]

# Existing key 'shared' keeps its original position even though its value changed.
merged_union


{'first': 1, 'shared': 'right', 'last': 3, 'new': 4}

In [4]:
# Shallow-copy identity refresher
original = {"numbers": [1, 2], "meta": {"active": True}}
shallow = original.copy()
deep = deepcopy(original)

identity_facts = {
    "top_level_shallow_is_new": shallow is not original,
    "shallow_list_is_shared": shallow["numbers"] is original["numbers"],
    "shallow_nested_dict_is_shared": shallow["meta"] is original["meta"],
    "deep_list_is_new": deep["numbers"] is not original["numbers"],
    "deep_nested_dict_is_new": deep["meta"] is not original["meta"],
}

assert all(identity_facts.values())
identity_facts


{'top_level_shallow_is_new': True,
 'shallow_list_is_shared': True,
 'shallow_nested_dict_is_shared': True,
 'deep_list_is_new': True,
 'deep_nested_dict_is_new': True}

## Problem 1 — Layered merge with provenance

Applications commonly combine configuration layers such as defaults, environment settings, and command-line overrides.

Implement `merge_layers(*named_layers)` where every argument is a `(name, mapping)` pair.

### Requirements

- Return `(merged, provenance)`.
- Later layers win on key collisions.
- Do not mutate any input mapping.
- `provenance[key]` must contain the name of the layer that supplied the final value.
- Preserve normal dictionary insertion-order behavior: the first appearance of a key determines its position.

### Example

```python
defaults = {"host": "localhost", "port": 5432, "debug": False}
environment = {"host": "db.internal", "debug": True}
cli = {"debug": False, "workers": 8}
```

The final order should be `host, port, debug, workers`, even though `host` and `debug` are overwritten.


In [5]:
# Starter signature

def merge_layers(*named_layers: tuple[str, Mapping[str, Any]]) -> tuple[dict[str, Any], dict[str, str]]:
    raise NotImplementedError


### Solution

A normal assignment or `update` replaces the value of an existing key without relocating the key. Updating provenance in the same loop records the final winning layer.


In [6]:
def merge_layers(*named_layers: tuple[str, Mapping[str, Any]]) -> tuple[dict[str, Any], dict[str, str]]:
    merged: dict[str, Any] = {}
    provenance: dict[str, str] = {}

    for layer_name, layer in named_layers:
        merged.update(layer)
        for key in layer:
            provenance[key] = layer_name

    return merged, provenance


defaults = {"host": "localhost", "port": 5432, "debug": False}
environment = {"host": "db.internal", "debug": True}
cli = {"debug": False, "workers": 8}

merged, provenance = merge_layers(
    ("defaults", defaults),
    ("environment", environment),
    ("cli", cli),
)

assert merged == {
    "host": "db.internal",
    "port": 5432,
    "debug": False,
    "workers": 8,
}
assert list(merged) == ["host", "port", "debug", "workers"]
assert provenance == {
    "host": "environment",
    "port": "defaults",
    "debug": "cli",
    "workers": "cli",
}
assert defaults == {"host": "localhost", "port": 5432, "debug": False}
assert environment == {"host": "db.internal", "debug": True}
assert cli == {"debug": False, "workers": 8}

merged, provenance


({'host': 'db.internal', 'port': 5432, 'debug': False, 'workers': 8},
 {'host': 'environment', 'port': 'defaults', 'debug': 'cli', 'workers': 'cli'})

## Problem 2 — Strict merge with collision detection

Sometimes "last value wins" is dangerous. A duplicated key may indicate a configuration mistake.

Implement `strict_merge(*mappings, allowed_overrides=frozenset())`.

### Requirements

- Return a new merged dictionary.
- Reject every duplicate key unless it appears in `allowed_overrides`.
- Report all illegal collisions, not just the first one.
- Do not mutate inputs.
- Keep ordinary insertion order when the merge is valid.

Create a custom `DuplicateKeyError` whose `.collisions` attribute is a dictionary mapping each duplicated key to the zero-based mapping indexes where it appeared.


In [7]:
class DuplicateKeyError(ValueError):
    pass


def strict_merge(
    *mappings: Mapping[Any, Any],
    allowed_overrides: frozenset[Any] = frozenset(),
) -> dict[Any, Any]:
    raise NotImplementedError


### Solution

Validate collisions first, then construct the result. Separating validation from construction makes the behavior easier to reason about and provides a complete error report.


In [8]:
class DuplicateKeyError(ValueError):
    def __init__(self, collisions: dict[Any, tuple[int, ...]]) -> None:
        self.collisions = collisions
        details = ", ".join(f"{key!r} in layers {indexes}" for key, indexes in collisions.items())
        super().__init__(f"Illegal duplicate keys: {details}")


def strict_merge(
    *mappings: Mapping[Any, Any],
    allowed_overrides: frozenset[Any] = frozenset(),
) -> dict[Any, Any]:
    occurrences: dict[Any, list[int]] = {}

    for index, mapping in enumerate(mappings):
        for key in mapping:
            occurrences.setdefault(key, []).append(index)

    collisions = {
        key: tuple(indexes)
        for key, indexes in occurrences.items()
        if len(indexes) > 1 and key not in allowed_overrides
    }

    if collisions:
        raise DuplicateKeyError(collisions)

    result: dict[Any, Any] = {}
    for mapping in mappings:
        result.update(mapping)
    return result


base = {"host": "localhost", "port": 5432}
team = {"port": 6432, "pool": 10}
user = {"host": "127.0.0.1", "timeout": 5}

try:
    strict_merge(base, team, user)
except DuplicateKeyError as exc:
    assert exc.collisions == {"host": (0, 2), "port": (0, 1)}
else:
    raise AssertionError("Expected DuplicateKeyError")

valid = strict_merge(base, team, user, allowed_overrides=frozenset({"host", "port"}))
assert valid == {"host": "127.0.0.1", "port": 6432, "pool": 10, "timeout": 5}
assert list(valid) == ["host", "port", "pool", "timeout"]
valid


{'host': '127.0.0.1', 'port': 6432, 'pool': 10, 'timeout': 5}

## Problem 3 — Non-mutating recursive merge with policies

A shallow merge replaces an entire nested dictionary. For configuration data, a recursive merge is often more useful.

Implement:

```python
deep_merge(left, right, *, list_policy="replace", on_type_conflict="overwrite")
```

### Rules

- If both values are mappings, merge recursively.
- If both values are lists:
  - `list_policy="replace"`: use the right list;
  - `list_policy="extend"`: concatenate the lists.
- For all other collisions, the right value wins.
- With `on_type_conflict="error"`, raise `TypeError` when one side is a mapping/list and the other side has an incompatible type.
- The returned structure must not share mutable nested containers with either input.
- Neither input may be mutated.


In [9]:
def deep_merge(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    *,
    list_policy: str = "replace",
    on_type_conflict: str = "overwrite",
) -> dict[str, Any]:
    raise NotImplementedError


### Solution

`deepcopy` at the assignment boundaries prevents aliases to either input. Recursive calls handle mapping collisions. Explicit policy validation avoids silently accepting misspelled policy names.


In [10]:
def deep_merge(
    left: Mapping[str, Any],
    right: Mapping[str, Any],
    *,
    list_policy: str = "replace",
    on_type_conflict: str = "overwrite",
) -> dict[str, Any]:
    if list_policy not in {"replace", "extend"}:
        raise ValueError("list_policy must be 'replace' or 'extend'")
    if on_type_conflict not in {"overwrite", "error"}:
        raise ValueError("on_type_conflict must be 'overwrite' or 'error'")

    result = deepcopy(dict(left))

    for key, right_value in right.items():
        if key not in result:
            result[key] = deepcopy(right_value)
            continue

        left_value = result[key]

        if isinstance(left_value, Mapping) and isinstance(right_value, Mapping):
            result[key] = deep_merge(
                left_value,
                right_value,
                list_policy=list_policy,
                on_type_conflict=on_type_conflict,
            )
        elif isinstance(left_value, list) and isinstance(right_value, list):
            if list_policy == "replace":
                result[key] = deepcopy(right_value)
            else:
                result[key] = deepcopy(left_value) + deepcopy(right_value)
        else:
            structured_left = isinstance(left_value, (Mapping, list))
            structured_right = isinstance(right_value, (Mapping, list))
            if on_type_conflict == "error" and structured_left != structured_right:
                raise TypeError(
                    f"Type conflict at key {key!r}: "
                    f"{type(left_value).__name__} vs {type(right_value).__name__}"
                )
            result[key] = deepcopy(right_value)

    return result


left = {
    "db": {"host": "localhost", "options": {"ssl": False, "retries": 2}},
    "features": ["search"],
    "mode": "development",
}
right = {
    "db": {"options": {"ssl": True}, "pool": 20},
    "features": ["payments"],
    "mode": "production",
}

replaced = deep_merge(left, right)
extended = deep_merge(left, right, list_policy="extend")

assert replaced == {
    "db": {
        "host": "localhost",
        "options": {"ssl": True, "retries": 2},
        "pool": 20,
    },
    "features": ["payments"],
    "mode": "production",
}
assert extended["features"] == ["search", "payments"]

# No mutable aliases to either input.
assert replaced["db"] is not left["db"]
assert replaced["db"]["options"] is not right["db"]["options"]
assert replaced["features"] is not right["features"]

# Inputs remain unchanged.
assert left["db"]["options"]["ssl"] is False
assert right["db"]["options"] == {"ssl": True}

try:
    deep_merge({"service": {"port": 80}}, {"service": "disabled"}, on_type_conflict="error")
except TypeError as exc:
    assert "service" in str(exc)
else:
    raise AssertionError("Expected a type conflict")

replaced


{'db': {'host': 'localhost',
  'options': {'ssl': True, 'retries': 2},
  'pool': 20},
 'features': ['payments'],
 'mode': 'production'}

## Problem 4 — Recursive patching with a deletion sentinel

Configuration patches often need three distinct operations:

- add a key;
- replace a value;
- delete a key.

`None` cannot reliably mean "delete" because `None` may be a legitimate value. Use a unique sentinel instead.

Implement `apply_patch(document, patch, *, strict_delete=True)`.

### Rules

- A mapping value patches a nested mapping recursively.
- `DELETE` removes the corresponding key.
- Other values replace the old value.
- Return a deep, independent result.
- Never mutate `document` or `patch`.
- If `strict_delete=True`, deleting a missing key raises `KeyError`.
- The update must be transactional: an error must not partially modify the original document.


In [11]:
DELETE = object()


def apply_patch(
    document: Mapping[str, Any],
    patch: Mapping[str, Any],
    *,
    strict_delete: bool = True,
) -> dict[str, Any]:
    raise NotImplementedError


### Solution

Work on a deep copy. Any failure affects only the temporary result, so the original remains untouched.


In [12]:
DELETE = object()


def apply_patch(
    document: Mapping[str, Any],
    patch: Mapping[str, Any],
    *,
    strict_delete: bool = True,
) -> dict[str, Any]:
    result = deepcopy(dict(document))

    for key, patch_value in patch.items():
        if patch_value is DELETE:
            if key not in result:
                if strict_delete:
                    raise KeyError(f"Cannot delete missing key {key!r}")
                continue
            del result[key]
            continue

        current = result.get(key)
        if isinstance(current, Mapping) and isinstance(patch_value, Mapping):
            result[key] = apply_patch(current, patch_value, strict_delete=strict_delete)
        else:
            result[key] = deepcopy(patch_value)

    return result


document = {
    "user": {
        "name": "Ada",
        "email": "ada@example.test",
        "preferences": {"theme": "light", "alerts": True},
    },
    "roles": ["reader"],
    "temporary": True,
}
patch = {
    "user": {
        "email": None,                    # None is a real replacement value.
        "preferences": {"theme": "dark", "alerts": DELETE},
    },
    "roles": ["reader", "editor"],
    "temporary": DELETE,
}

patched = apply_patch(document, patch)

assert patched == {
    "user": {
        "name": "Ada",
        "email": None,
        "preferences": {"theme": "dark"},
    },
    "roles": ["reader", "editor"],
}
assert document["temporary"] is True
assert document["user"]["preferences"]["alerts"] is True
assert patched["roles"] is not patch["roles"]

try:
    apply_patch(document, {"missing": DELETE})
except KeyError:
    pass
else:
    raise AssertionError("Expected strict deletion to fail")

patched


{'user': {'name': 'Ada', 'email': None, 'preferences': {'theme': 'dark'}},
 'roles': ['reader', 'editor']}

## Problem 5 — Shallow-copy mutation detective

Without running the code first, predict every final value and identity relationship.

```python
source = {
    "numbers": [1, 2],
    "profile": {"name": "Lin", "tags": ["python"]},
}

copy_a = source.copy()
copy_b = {**source}
copy_c = dict(source)
copy_d = deepcopy(source)

copy_a["numbers"].append(3)
copy_b["profile"]["name"] = "Grace"
copy_c["profile"]["tags"] = ["data"]
copy_d["numbers"].append(99)
copy_d["profile"]["tags"].append("deep")
```

Questions:

1. What are the final values of `source`, `copy_a`, `copy_b`, `copy_c`, and `copy_d`?
2. Which top-level dictionaries are identical objects?
3. Which nested objects are shared?
4. Why does replacing `copy_c["profile"]["tags"]` affect the other shallow copies?


### Solution

All three shallow-copy techniques create a new top-level dictionary but retain references to the same nested list and nested dictionary. `deepcopy` recursively creates independent mutable containers.


In [13]:
source = {
    "numbers": [1, 2],
    "profile": {"name": "Lin", "tags": ["python"]},
}

copy_a = source.copy()
copy_b = {**source}
copy_c = dict(source)
copy_d = deepcopy(source)

copy_a["numbers"].append(3)
copy_b["profile"]["name"] = "Grace"
copy_c["profile"]["tags"] = ["data"]
copy_d["numbers"].append(99)
copy_d["profile"]["tags"].append("deep")

assert source == {
    "numbers": [1, 2, 3],
    "profile": {"name": "Grace", "tags": ["data"]},
}
assert copy_a == source == copy_b == copy_c
assert copy_d == {
    "numbers": [1, 2, 99],
    "profile": {"name": "Lin", "tags": ["python", "deep"]},
}

identity_report = {
    "all_top_level_dicts_distinct": len({id(source), id(copy_a), id(copy_b), id(copy_c), id(copy_d)}) == 5,
    "shallow_numbers_shared": source["numbers"] is copy_a["numbers"] is copy_b["numbers"] is copy_c["numbers"],
    "shallow_profile_shared": source["profile"] is copy_a["profile"] is copy_b["profile"] is copy_c["profile"],
    "deep_numbers_independent": copy_d["numbers"] is not source["numbers"],
    "deep_profile_independent": copy_d["profile"] is not source["profile"],
}

assert all(identity_report.values())
identity_report


{'all_top_level_dicts_distinct': True,
 'shallow_numbers_shared': True,
 'shallow_profile_shared': True,
 'deep_numbers_independent': True,
 'deep_profile_independent': True}

## Problem 6 — Copy-on-write update for a nested path

A full `deepcopy` can be wasteful when changing only one path in a large structure. Copy-on-write creates new dictionaries only along the modified path and shares untouched branches.

Implement:

```python
cow_set(document, path, value)
```

### Requirements

- `path` is a non-empty tuple of dictionary keys.
- Every intermediate path component must already exist and must be a mapping.
- Return a new top-level dictionary.
- Do not mutate the original.
- Copy only dictionaries along the path.
- Share untouched branches by identity.
- Store a deep copy of the new value so later mutations of the caller's value do not affect the result.


In [14]:
def cow_set(document: Mapping[str, Any], path: tuple[str, ...], value: Any) -> dict[str, Any]:
    raise NotImplementedError


### Solution

Recursively shallow-copy the current mapping, replace one child with the recursively updated child, and return. This is the central idea behind persistent data structures.


In [15]:
def cow_set(document: Mapping[str, Any], path: tuple[str, ...], value: Any) -> dict[str, Any]:
    if not path:
        raise ValueError("path must contain at least one key")

    key, *remaining = path
    result = dict(document)

    if not remaining:
        result[key] = deepcopy(value)
        return result

    if key not in document:
        raise KeyError(f"Missing intermediate key {key!r}")
    child = document[key]
    if not isinstance(child, Mapping):
        raise TypeError(f"Intermediate value at {key!r} is not a mapping")

    result[key] = cow_set(child, tuple(remaining), value)
    return result


state = {
    "account": {
        "name": "Mina",
        "preferences": {"theme": "light", "density": "comfortable"},
    },
    "activity": {"last_login": "2026-08-01", "events": [1, 2, 3]},
}
new_value = {"name": "dark", "contrast": "high"}
updated = cow_set(state, ("account", "preferences", "theme"), new_value)

assert state["account"]["preferences"]["theme"] == "light"
assert updated["account"]["preferences"]["theme"] == new_value

# Copied path:
assert updated is not state
assert updated["account"] is not state["account"]
assert updated["account"]["preferences"] is not state["account"]["preferences"]

# Untouched branch is intentionally shared:
assert updated["activity"] is state["activity"]

# Stored value is independent from the caller's object:
new_value["contrast"] = "low"
assert updated["account"]["preferences"]["theme"]["contrast"] == "high"

updated


{'account': {'name': 'Mina',
  'preferences': {'theme': {'name': 'dark', 'contrast': 'high'},
   'density': 'comfortable'}},
 'activity': {'last_login': '2026-08-01', 'events': [1, 2, 3]}}

## Problem 7 — Transactional validated in-place update

A normal `dict.update` can leave a dictionary in an invalid state if validation happens afterward.

Implement:

```python
validated_update(target, updates, validators)
```

where `validators` maps keys to callables of the form:

```python
validator(value, complete_candidate) -> None
```

A validator returns normally when valid and raises an exception when invalid.

### Requirements

- Validate a candidate dictionary containing all proposed updates.
- If any validator fails, leave `target` completely unchanged.
- If validation succeeds, update the original `target` in place and return it.
- Run validators only for keys present in `validators`.
- Allow cross-field validation through `complete_candidate`.


In [16]:
def validated_update(
    target: dict[str, Any],
    updates: Mapping[str, Any],
    validators: Mapping[str, Callable[[Any, Mapping[str, Any]], None]],
) -> dict[str, Any]:
    raise NotImplementedError


### Solution

Build and validate a temporary candidate first. Commit with `clear` and `update` only after every validator succeeds. Using `clear` plus `update` also handles key removal if the candidate-building policy is later extended.


In [17]:
def validated_update(
    target: dict[str, Any],
    updates: Mapping[str, Any],
    validators: Mapping[str, Callable[[Any, Mapping[str, Any]], None]],
) -> dict[str, Any]:
    candidate = target.copy()
    candidate.update(updates)

    for key, validator in validators.items():
        if key in candidate:
            validator(candidate[key], candidate)

    target.clear()
    target.update(candidate)
    return target


def validate_port(value: Any, _: Mapping[str, Any]) -> None:
    if not isinstance(value, int) or isinstance(value, bool) or not 1 <= value <= 65535:
        raise ValueError("port must be an integer from 1 to 65535")


def validate_timeout(value: Any, _: Mapping[str, Any]) -> None:
    if not isinstance(value, (int, float)) or isinstance(value, bool) or value <= 0:
        raise ValueError("timeout must be positive")


def validate_tls_port(_: Any, candidate: Mapping[str, Any]) -> None:
    if candidate.get("tls") is True and candidate.get("port") == 80:
        raise ValueError("TLS cannot use the clear-text port 80")


settings = {"host": "localhost", "port": 8080, "timeout": 2.0, "tls": False}
validators = {
    "port": validate_port,
    "timeout": validate_timeout,
    "tls": validate_tls_port,
}

before = settings.copy()
try:
    validated_update(settings, {"port": 80, "tls": True}, validators)
except ValueError as exc:
    assert "TLS" in str(exc)
else:
    raise AssertionError("Expected validation failure")
assert settings == before  # Transaction preserved.

validated_update(settings, {"port": 443, "tls": True, "timeout": 5.0}, validators)
assert settings == {"host": "localhost", "port": 443, "timeout": 5.0, "tls": True}
settings


{'host': 'localhost', 'port': 443, 'timeout': 5.0, 'tls': True}

## Problem 8 — A safer transactional wrapper around `dict.update`

Reproduce the three common `dict.update` input forms while guaranteeing that malformed input cannot partially modify the target.

Implement:

```python
safe_update(target, source=None, /, **kwargs)
```

### Requirements

- Accept a mapping, an iterable of two-item iterables, keyword arguments, or a combination of source plus keywords.
- Match normal last-value-wins behavior.
- Return the original `target` after mutating it successfully.
- If the input is malformed, raise `UpdateInputError` and leave `target` unchanged.
- Preserve ordinary insertion-order behavior.

A simple and robust approach is to let the built-in `dict.update` validate the input on a temporary candidate.


In [18]:
class UpdateInputError(ValueError):
    pass


def safe_update(target: dict[Any, Any], source: Any = None, /, **kwargs: Any) -> dict[Any, Any]:
    raise NotImplementedError


### Solution

The built-in implementation already handles mappings, pair iterables, duplicate pairs, and keyword arguments correctly. Reuse it on a temporary dictionary, then commit.


In [19]:
class UpdateInputError(ValueError):
    pass


_MISSING = object()


def safe_update(target: dict[Any, Any], source: Any = _MISSING, /, **kwargs: Any) -> dict[Any, Any]:
    candidate = target.copy()

    try:
        if source is _MISSING:
            candidate.update(**kwargs)
        else:
            candidate.update(source, **kwargs)
    except (TypeError, ValueError) as exc:
        raise UpdateInputError(f"Invalid update input: {exc}") from exc

    target.clear()
    target.update(candidate)
    return target


target = {"a": 1, "b": 2}
safe_update(target, [("b", 20), ("c", 30), ("c", 300)], d=400)
assert target == {"a": 1, "b": 20, "c": 300, "d": 400}
assert list(target) == ["a", "b", "c", "d"]

snapshot = target.copy()
try:
    safe_update(target, [("valid", 1), ("broken", 2, 3)])
except UpdateInputError:
    pass
else:
    raise AssertionError("Expected malformed pair input to fail")
assert target == snapshot

target


{'a': 1, 'b': 20, 'c': 300, 'd': 400}

## Problem 9 — Safely route a configuration dictionary into a function

Blindly calling `func(**config)` may fail because of unknown keys or missing required parameters.

Implement:

```python
call_with_config(func, config, *, strict=True)
```

### Requirements

- Inspect the callable's signature.
- Pass positional-or-keyword and keyword-only parameters from `config` as keyword arguments.
- Respect default values by omitting absent optional parameters.
- Raise `TypeError` with a clear list of missing required parameters.
- In strict mode, reject unknown keys unless the function accepts `**kwargs`.
- In non-strict mode, ignore unknown keys unless `**kwargs` is accepted; when accepted, forward them.
- Return the function's result.

For this exercise, functions with positional-only parameters are out of scope and should be rejected.


In [20]:
def call_with_config(func: Callable[..., Any], config: Mapping[str, Any], *, strict: bool = True) -> Any:
    raise NotImplementedError


### Solution

`inspect.signature` exposes parameter kinds, defaults, and whether `**kwargs` is present. Separate required-key validation from unknown-key handling for clearer errors.


In [21]:
def call_with_config(func: Callable[..., Any], config: Mapping[str, Any], *, strict: bool = True) -> Any:
    sig = signature(func)
    parameters = sig.parameters

    positional_only = [
        name for name, parameter in parameters.items()
        if parameter.kind is Parameter.POSITIONAL_ONLY
    ]
    if positional_only:
        raise TypeError(f"Positional-only parameters are unsupported: {positional_only}")

    accepts_var_kwargs = any(
        parameter.kind is Parameter.VAR_KEYWORD
        for parameter in parameters.values()
    )

    accepted_names = {
        name
        for name, parameter in parameters.items()
        if parameter.kind in {Parameter.POSITIONAL_OR_KEYWORD, Parameter.KEYWORD_ONLY}
    }

    required_names = {
        name
        for name, parameter in parameters.items()
        if parameter.kind in {Parameter.POSITIONAL_OR_KEYWORD, Parameter.KEYWORD_ONLY}
        and parameter.default is Parameter.empty
    }

    missing = sorted(required_names - config.keys())
    if missing:
        raise TypeError(f"Missing required configuration keys: {missing}")

    unknown = sorted(config.keys() - accepted_names)
    if strict and unknown and not accepts_var_kwargs:
        raise TypeError(f"Unknown configuration keys: {unknown}")

    if accepts_var_kwargs:
        forwarded = dict(config)
    else:
        forwarded = {key: config[key] for key in accepted_names if key in config}

    return func(**forwarded)


def connect(*, host: str, port: int = 5432, ssl: bool = False) -> str:
    return f"{host}:{port}?ssl={str(ssl).lower()}"


config = {"host": "db.internal", "ssl": True, "unused": "ignored"}

try:
    call_with_config(connect, config)
except TypeError as exc:
    assert "unused" in str(exc)
else:
    raise AssertionError("Strict mode should reject unknown keys")

assert call_with_config(connect, config, strict=False) == "db.internal:5432?ssl=true"

try:
    call_with_config(connect, {"port": 5432})
except TypeError as exc:
    assert "host" in str(exc)
else:
    raise AssertionError("Missing required key should fail")


def collector(*, required: int, **extras: Any) -> tuple[int, dict[str, Any]]:
    return required, extras

assert call_with_config(collector, {"required": 1, "x": 2, "y": 3}) == (1, {"x": 2, "y": 3})

call_with_config(connect, config, strict=False)


'db.internal:5432?ssl=true'

## Problem 10 — Merge with an audit trail

For debugging, it is useful to know not only the final dictionary but also how each layer changed it.

Implement `merge_with_audit(*named_layers, include_noops=False)`.

### Return value

```python
(final_dictionary, events)
```

Each event is a dictionary with:

- `key`;
- `source`;
- `action`: `"add"`, `"overwrite"`, or `"noop"`;
- `old`;
- `new`.

### Rules

- A new key creates an `add` event.
- A different replacement creates an `overwrite` event.
- Repeating an equal value creates a `noop` only when `include_noops=True`.
- Later layers win.
- Inputs are not mutated.


In [22]:
def merge_with_audit(
    *named_layers: tuple[str, Mapping[str, Any]],
    include_noops: bool = False,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    raise NotImplementedError


### Solution

Test membership before assignment so that a missing key with value `None` is distinguishable from an existing key whose value is `None`.


In [23]:
def merge_with_audit(
    *named_layers: tuple[str, Mapping[str, Any]],
    include_noops: bool = False,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    result: dict[str, Any] = {}
    events: list[dict[str, Any]] = []

    for source, layer in named_layers:
        for key, new_value in layer.items():
            if key not in result:
                action = "add"
                old_value = None
            elif result[key] == new_value:
                action = "noop"
                old_value = result[key]
            else:
                action = "overwrite"
                old_value = result[key]

            if action != "noop" or include_noops:
                events.append(
                    {
                        "key": key,
                        "source": source,
                        "action": action,
                        "old": deepcopy(old_value),
                        "new": deepcopy(new_value),
                    }
                )

            result[key] = deepcopy(new_value)

    return result, events


final_config, events = merge_with_audit(
    ("defaults", {"port": 5432, "debug": False, "region": None}),
    ("environment", {"debug": True, "region": None}),
    ("cli", {"port": 6432}),
    include_noops=True,
)

assert final_config == {"port": 6432, "debug": True, "region": None}
assert [event["action"] for event in events] == [
    "add", "add", "add", "overwrite", "noop", "overwrite"
]
assert events[-1] == {
    "key": "port",
    "source": "cli",
    "action": "overwrite",
    "old": 5432,
    "new": 6432,
}

events


[{'key': 'port',
  'source': 'defaults',
  'action': 'add',
  'old': None,
  'new': 5432},
 {'key': 'debug',
  'source': 'defaults',
  'action': 'add',
  'old': None,
  'new': False},
 {'key': 'region',
  'source': 'defaults',
  'action': 'add',
  'old': None,
  'new': None},
 {'key': 'debug',
  'source': 'environment',
  'action': 'overwrite',
  'old': False,
  'new': True},
 {'key': 'region',
  'source': 'environment',
  'action': 'noop',
  'old': None,
  'new': None},
 {'key': 'port',
  'source': 'cli',
  'action': 'overwrite',
  'old': 5432,
  'new': 6432}]

## Problem 11 — Deep-copy graph topology: cycles and shared references

`deepcopy` does more than recursively copy containers. It also preserves the topology of an object graph by using an internal memo table.

Construct a dictionary with:

- a self-reference through key `"self"`;
- one shared list referenced by both `"primary"` and `"backup"`.

Deep-copy it and verify all of the following:

1. the top-level clone is a different object;
2. `clone["self"] is clone`;
3. the copied shared list is different from the original shared list;
4. `clone["primary"] is clone["backup"]`;
5. mutating the copied shared list does not affect the original.

Explain why a naive recursive copier without memoization would fail.


### Solution

A naive copier would recurse forever on the self-reference. Even if cycle detection were added without memoized reuse, it could accidentally duplicate one shared child into two separate children. `deepcopy` records already-copied object identities and reuses the corresponding clone.


In [24]:
shared = ["event-1"]
graph: dict[str, Any] = {
    "primary": shared,
    "backup": shared,
}
graph["self"] = graph

clone = deepcopy(graph)

assert clone is not graph
assert clone["self"] is clone
assert clone["primary"] is not shared
assert clone["primary"] is clone["backup"]

clone["primary"].append("event-2")
assert shared == ["event-1"]
assert clone["backup"] == ["event-1", "event-2"]

{
    "clone_is_new": clone is not graph,
    "cycle_points_to_clone": clone["self"] is clone,
    "shared_child_preserved": clone["primary"] is clone["backup"],
    "original_child_isolated": clone["primary"] is not shared,
}


{'clone_is_new': True,
 'cycle_points_to_clone': True,
 'shared_child_preserved': True,
 'original_child_isolated': True}

## Problem 12 — Custom deep-copy behavior for dictionary-backed state

Sometimes a state object contains both ordinary mutable data and a resource that must not be duplicated, such as a connection pool.

Create:

- a `Connection` object;
- an `AppState` object whose `.data` attribute is a dictionary and whose `.connection` attribute refers to a `Connection`;
- a custom `AppState.__deepcopy__` method that deep-copies `.data` but intentionally shares `.connection`.

### Requirements

- Respect the `memo` dictionary to support repeated references and cycles.
- Register the new `AppState` object in `memo` before recursively copying `.data`.
- Demonstrate that nested state data is independent while the connection is shared.


In [25]:
@dataclass
class Connection:
    name: str


class AppState:
    def __init__(self, data: dict[str, Any], connection: Connection) -> None:
        self.data = data
        self.connection = connection

    def __deepcopy__(self, memo: dict[int, Any]) -> "AppState":
        raise NotImplementedError


### Solution

Registering the partially constructed result in `memo` before copying nested state is the key best practice. It prevents infinite recursion if `.data` eventually points back to the `AppState`.


In [26]:
@dataclass
class Connection:
    name: str


class AppState:
    def __init__(self, data: dict[str, Any], connection: Connection) -> None:
        self.data = data
        self.connection = connection

    def __deepcopy__(self, memo: dict[int, Any]) -> "AppState":
        existing = memo.get(id(self))
        if existing is not None:
            return existing

        result = type(self).__new__(type(self))
        memo[id(self)] = result
        result.connection = self.connection                 # Intentionally shared.
        result.data = deepcopy(self.data, memo)             # Recursively independent.
        return result


connection = Connection("primary-db")
state = AppState(
    data={"cache": {"users": [1, 2]}, "flags": {"beta": False}},
    connection=connection,
)
state_copy = deepcopy(state)

assert state_copy is not state
assert state_copy.data is not state.data
assert state_copy.data["cache"] is not state.data["cache"]
assert state_copy.data["cache"]["users"] is not state.data["cache"]["users"]
assert state_copy.connection is state.connection

state_copy.data["cache"]["users"].append(3)
assert state.data["cache"]["users"] == [1, 2]

{
    "data_independent": state_copy.data is not state.data,
    "connection_shared": state_copy.connection is state.connection,
}


{'data_independent': True, 'connection_shared': True}

## Problem 13 — Benchmark shallow-copy techniques responsibly

Compare these shallow-copy techniques:

- `d.copy()`;
- `dict(d)`;
- `{**d}`;
- `{key: value for key, value in d.items()}`.

### Best-practice requirements

- Verify semantic equivalence before timing.
- Build input data outside the timed callable.
- Benchmark more than one dictionary size.
- Use repeated measurements and report the median.
- Keep the benchmark small enough to run interactively.
- Do not claim that one result is universally fastest; timings depend on Python version, hardware, and data shape.

Implement `benchmark_copy_methods(sizes=(100, 10_000), repeats=5, number=200)` and return a list of result dictionaries.


In [27]:
def benchmark_copy_methods(
    sizes: tuple[int, ...] = (100, 10_000),
    repeats_count: int = 5,
    number: int = 200,
) -> list[dict[str, Any]]:
    raise NotImplementedError


### Solution

`timeit.repeat` reduces one-off noise, and the median is less sensitive to outliers than the minimum or arithmetic mean. A comprehension performs Python-level iteration and is usually slower for a plain copy, but measure rather than assume.


In [28]:
def benchmark_copy_methods(
    sizes: tuple[int, ...] = (100, 10_000),
    repeats_count: int = 5,
    number: int = 200,
) -> list[dict[str, Any]]:
    methods: dict[str, Callable[[dict[int, list[int]]], dict[int, list[int]]]] = {
        "copy": lambda d: d.copy(),
        "dict_constructor": lambda d: dict(d),
        "unpacking": lambda d: {**d},
        "comprehension": lambda d: {key: value for key, value in d.items()},
    }

    results: list[dict[str, Any]] = []

    for size in sizes:
        data = {index: [index] for index in range(size)}

        # Correctness and shallow-copy semantics are checked separately from timing.
        for method_name, method in methods.items():
            copied = method(data)
            assert copied == data
            assert copied is not data
            if data:
                first_key = next(iter(data))
                assert copied[first_key] is data[first_key], f"{method_name} was not shallow"

        for method_name, method in methods.items():
            samples = repeat(lambda: method(data), repeat=repeats_count, number=number)
            results.append(
                {
                    "size": size,
                    "method": method_name,
                    "median_seconds": median(samples),
                    "runs_per_sample": number,
                }
            )

    return results


benchmark_results = benchmark_copy_methods(sizes=(100, 5_000), repeats_count=3, number=50)

# Display rounded values without changing the stored measurements.
for row in benchmark_results:
    print(
        f"size={row['size']:>5}  method={row['method']:<16}  "
        f"median={row['median_seconds']:.6f}s for {row['runs_per_sample']} copies"
    )


size=  100  method=copy              median=0.000029s for 50 copies
size=  100  method=dict_constructor  median=0.000031s for 50 copies
size=  100  method=unpacking         median=0.000029s for 50 copies
size=  100  method=comprehension     median=0.000372s for 50 copies
size= 5000  method=copy              median=0.002244s for 50 copies
size= 5000  method=dict_constructor  median=0.002546s for 50 copies
size= 5000  method=unpacking         median=0.002876s for 50 copies
size= 5000  method=comprehension     median=0.017839s for 50 copies


## Problem 14 — Capstone: production-style configuration resolver

Build a resolver that combines multiple ideas from the notebook.

Implement:

```python
resolve_configuration(*named_layers, schema)
```

### Layer behavior

- Each layer is a `(name, mapping)` pair.
- Nested mappings merge recursively.
- Non-mapping collisions use the later value.
- Inputs are never mutated.
- The final structure must not share mutable containers with inputs.

### Provenance

Return provenance for every leaf using tuple paths:

```python
("database", "host") -> "environment"
```

When an entire subtree is replaced by a scalar or list, remove stale provenance entries below that path.

### Validation

`schema` maps leaf paths to validator functions. Each validator receives the final leaf value and raises on invalid input.

- Every schema path must exist.
- Validation happens before the result is returned.

### Immutability

Return a recursively frozen result:

- dictionaries become `MappingProxyType`;
- lists become tuples;
- tuples are recursively frozen;
- other values are returned unchanged.

### Return value

```python
(frozen_config, provenance)
```


In [29]:
def resolve_configuration(
    *named_layers: tuple[str, Mapping[str, Any]],
    schema: Mapping[tuple[str, ...], Callable[[Any], None]],
) -> tuple[Mapping[str, Any], dict[tuple[str, ...], str]]:
    raise NotImplementedError


### Solution

The resolver separates four concerns:

1. recursively overlay each layer;
2. update leaf provenance;
3. validate the fully merged mutable result;
4. freeze the validated result.

Keeping these phases separate improves testability and prevents partially valid outputs.


In [30]:
def _leaf_paths(value: Any, prefix: tuple[str, ...]) -> Iterable[tuple[str, ...]]:
    if isinstance(value, Mapping):
        if not value:
            yield prefix
        else:
            for key, child in value.items():
                yield from _leaf_paths(child, prefix + (str(key),))
    else:
        yield prefix


def _remove_provenance_subtree(
    provenance: dict[tuple[str, ...], str],
    prefix: tuple[str, ...],
) -> None:
    for path in [path for path in provenance if path[: len(prefix)] == prefix]:
        del provenance[path]


def _overlay_with_provenance(
    target: dict[str, Any],
    incoming: Mapping[str, Any],
    source: str,
    provenance: dict[tuple[str, ...], str],
    prefix: tuple[str, ...] = (),
) -> None:
    for key, incoming_value in incoming.items():
        key_str = str(key)
        path = prefix + (key_str,)
        current_value = target.get(key)

        if isinstance(current_value, Mapping) and isinstance(incoming_value, Mapping):
            # target contains ordinary dicts because all inserted mappings are deep-copied below.
            _overlay_with_provenance(target[key], incoming_value, source, provenance, path)
            continue

        _remove_provenance_subtree(provenance, path)
        target[key] = deepcopy(incoming_value)
        for leaf_path in _leaf_paths(incoming_value, path):
            provenance[leaf_path] = source


def _get_path(document: Mapping[str, Any], path: tuple[str, ...]) -> Any:
    current: Any = document
    for key in path:
        if not isinstance(current, Mapping) or key not in current:
            raise KeyError(f"Missing required configuration path: {path}")
        current = current[key]
    return current


def _freeze(value: Any) -> Any:
    if isinstance(value, Mapping):
        return MappingProxyType({key: _freeze(child) for key, child in value.items()})
    if isinstance(value, list):
        return tuple(_freeze(child) for child in value)
    if isinstance(value, tuple):
        return tuple(_freeze(child) for child in value)
    return value


def resolve_configuration(
    *named_layers: tuple[str, Mapping[str, Any]],
    schema: Mapping[tuple[str, ...], Callable[[Any], None]],
) -> tuple[Mapping[str, Any], dict[tuple[str, ...], str]]:
    merged: dict[str, Any] = {}
    provenance: dict[tuple[str, ...], str] = {}

    for source, layer in named_layers:
        _overlay_with_provenance(merged, layer, source, provenance)

    for path, validator in schema.items():
        validator(_get_path(merged, path))

    return _freeze(merged), provenance


def must_be_nonempty_string(value: Any) -> None:
    if not isinstance(value, str) or not value.strip():
        raise ValueError("expected a non-empty string")


def must_be_port(value: Any) -> None:
    if not isinstance(value, int) or isinstance(value, bool) or not 1 <= value <= 65535:
        raise ValueError("expected a valid TCP port")


def must_be_positive_int(value: Any) -> None:
    if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
        raise ValueError("expected a positive integer")


defaults = {
    "database": {
        "host": "localhost",
        "port": 5432,
        "options": {"ssl": False, "retries": 2},
    },
    "features": ["search"],
    "workers": 2,
}

environment = {
    "database": {
        "host": "db.internal",
        "options": {"ssl": True},
    },
    "features": ["search", "payments"],
}

cli = {
    "database": {"port": 6432},
    "workers": 8,
}

schema = {
    ("database", "host"): must_be_nonempty_string,
    ("database", "port"): must_be_port,
    ("workers",): must_be_positive_int,
}

frozen, provenance = resolve_configuration(
    ("defaults", defaults),
    ("environment", environment),
    ("cli", cli),
    schema=schema,
)

assert frozen["database"]["host"] == "db.internal"
assert frozen["database"]["port"] == 6432
assert frozen["database"]["options"]["ssl"] is True
assert frozen["database"]["options"]["retries"] == 2
assert frozen["features"] == ("search", "payments")
assert frozen["workers"] == 8

assert provenance[("database", "host")] == "environment"
assert provenance[("database", "port")] == "cli"
assert provenance[("database", "options", "ssl")] == "environment"
assert provenance[("database", "options", "retries")] == "defaults"
assert provenance[("features",)] == "environment"
assert provenance[("workers",)] == "cli"

# The configuration is recursively immutable.
try:
    frozen["workers"] = 20
except TypeError:
    pass
else:
    raise AssertionError("Top-level mapping should be immutable")

try:
    frozen["database"]["host"] = "other"
except TypeError:
    pass
else:
    raise AssertionError("Nested mapping should be immutable")

# Inputs remain untouched.
assert defaults["database"]["host"] == "localhost"
assert environment["database"]["options"] == {"ssl": True}

frozen, provenance


(mappingproxy({'database': mappingproxy({'host': 'db.internal',
                             'port': 6432,
                             'options': mappingproxy({'ssl': True,
                                           'retries': 2})}),
               'features': ('search', 'payments'),
               'workers': 8}),
 {('database', 'options', 'retries'): 'defaults',
  ('database', 'host'): 'environment',
  ('database', 'options', 'ssl'): 'environment',
  ('features',): 'environment',
  ('database', 'port'): 'cli',
  ('workers',): 'cli'})

## Additional challenge prompts

These are intentionally left without reference solutions so you can extend the notebook:

1. Add a `list_policy` to the capstone resolver with `replace`, `extend`, and `unique_extend` modes.
2. Add a conflict policy that allows overrides only from named trusted layers.
3. Add deletion support to the capstone using the `DELETE` sentinel.
4. Generate a human-readable diff from the capstone's provenance and audit data.
5. Support integer list indexes in `cow_set` while preserving copy-on-write behavior.
6. Make `strict_merge` report value pairs and layer names, not only indexes.
7. Add schema defaults that are inserted only when a path is absent.
8. Add a redaction function that deep-copies a nested configuration while replacing secret values at selected paths.
9. Benchmark `deepcopy` against targeted copy-on-write for different nesting depths.
10. Write property-based tests (for example with Hypothesis) asserting that no non-mutating merge changes its inputs.


## Summary of best practices

- Use direct assignment or `update` for intentional in-place changes.
- Use `d.copy()`, `dict(d)`, `{**d}`, or `d | {}` for a shallow top-level copy; choose based on readability and merge needs.
- Remember that all shallow-copy forms share nested mutable values.
- Use `deepcopy` when the entire mutable object graph must be independent.
- Prefer copy-on-write when only a small path changes and sharing untouched branches is acceptable.
- Validate before committing when an update must be transactional.
- Make collision policies explicit in reusable merge utilities.
- Track provenance or audit events when configuration layers are difficult to debug.
- Reuse built-in dictionary validation behavior instead of reimplementing pair parsing unnecessarily.
- Benchmark with repeated measurements, equivalent semantics, and realistic input sizes.
